<h1>Light GBM Model Build</h1>
<p>This script aims to train a common ML model - LGBM (Light Gradient Boosting Machine). This is an algorithm from Microsoft which uses tree based learning algorithms (like XGBoost), however it is known to be more efficient, faster and more accurate. You can read more about it at <a href="https://lightgbm.readthedocs.io/en/latest/index.html">lgbm website</a>. </p>

<h2>Import Libraries</h2>
<p>Only import required and used libraries in this script - removed unused libraries</p>

In [1]:
import os
import pandas as pd
import numpy as np
import statsmodels.base.model as Model
import joblib
import lightgbm as lgbm
from sklearn.model_selection import RandomizedSearchCV

<h2>Read Data</h2>
<p>Reading in raw data</p>

In [2]:
cwd = os.getcwd().split('amts')[0] + 'amts'
theme = "machine_learning"

In [3]:
input_folder_path = rf"{cwd}/{theme}/data/outputs/01_lgbm_introduction"

In [4]:
input_file_path = rf"{input_folder_path}/train_input_df.csv"
print(rf"Input File (Prepped Input Train Data): {input_file_path}")
train_input_df = pd.read_csv(input_file_path)

Input File (Prepped Input Train Data): c:\dev\amts/machine_learning/data/outputs/01_lgbm_introduction/train_input_df.csv


In [5]:
input_file_path = rf"{input_folder_path}/test_input_df.csv"
print(rf"Input File (Prepped Input Test Data): {input_file_path}")
test_input_df = pd.read_csv(input_file_path)

Input File (Prepped Input Test Data): c:\dev\amts/machine_learning/data/outputs/01_lgbm_introduction/test_input_df.csv


<h2>Light GBM Model Build</h2>

In [6]:
print('\n---Start Light GBM Model Build---')


---Start Light GBM Model Build---


<h3>Create base function for model build</h3>

In [7]:
class LightGbmModelBuilder:
    def __init__(self, input_train_data: pd.DataFrame, input_test_data: pd.DataFrame, independent_variables: list, target_variable: str, model_parameters: dict
                 ) -> {Model, pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame}:
        self.input_train_data = input_train_data
        self.input_test_data = input_test_data
        self.independent_variables = independent_variables
        self.target_variable = target_variable
        self.model_parameters = model_parameters

    def run(self):
        self.X_train, self.y_train, self.X_test, self.y_test = self.split_data_by_independent_and_target_variables()
        self.light_gbm_model = self.train_light_gbm_model()
        self.output_train_input_df, self.output_test_input_df = self.predict_target_variable()
        return self.light_gbm_model, self.X_train, self.y_train, self.X_test, self.y_test, self.output_train_input_df, self.output_test_input_df

    def split_data_by_independent_and_target_variables(self): 
        X_train = self.input_train_data[self.independent_variables].copy()
        y_train = np.array(self.input_train_data[[self.target_variable]].copy()).ravel()

        X_test = self.input_test_data[self.independent_variables].copy()
        y_test = np.array(self.input_test_data[[self.target_variable]].copy()).ravel()
        return X_train, y_train, X_test, y_test

    def train_light_gbm_model(self):
        self.model_parameters['random_state'] = 42
        light_gbm_model = lgbm.LGBMClassifier(**self.model_parameters)
        light_gbm_model.fit(self.X_train, self.y_train)
        return light_gbm_model

    def predict_target_variable(self):
        output_train_input_df = self.input_train_data.copy()
        output_test_input_df = self.input_test_data.copy()

        output_train_input_df['predicted_default_flag'] = self.light_gbm_model.predict_proba(self.X_train)[:, 1]
        output_test_input_df['predicted_default_flag'] = self.light_gbm_model.predict_proba(self.X_test)[:, 1]
        return output_train_input_df, output_test_input_df

<h3>Define target variable</h3>

In [8]:
target_variable_str = 'actual_default_flag'

<h3>Define independent variable</h3>

In [9]:
independent_variables_list = ['days_past_due', 'balance', 'credit_limit', 'external_score']

<h3>Base model</h3>

In [10]:
print('---Base Light GBM---')

---Base Light GBM---


<h4>Train model</h4>

In [11]:
model_parameters_dict = {'verbose': 2}
base_light_gbm_model, X_train, y_train, X_test, y_test, base_output_train_input_df, base_output_test_input_df = LightGbmModelBuilder(train_input_df, test_input_df, independent_variables_list, target_variable_str, model_parameters_dict).run()

[LightGBM] [Info] Number of positive: 1105, number of negative: 2220
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.082481
[LightGBM] [Debug] init for col-wise cost 0.000012 seconds, init for row-wise cost 0.000855 seconds
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001728 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 618
[LightGBM] [Info] Number of data points in the train set: 3325, number of used features: 4
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.332331 -> initscore=-0.697662
[LightGBM] [Info] Start training from score -0.697662
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 12
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 9
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 8
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 10
[LightGBM] [Debug] Trained a tree with leaves = 31 and depth = 9
[LightG

<h4>Get parameters</h4>

In [12]:
print('--Parameters--')
print(base_light_gbm_model.get_params())

--Parameters--
{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.1, 'max_depth': -1, 'min_child_samples': 20, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': None, 'num_leaves': 31, 'objective': None, 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbose': 2}


<h4>Compare average default flag between actual and predicted</h4>

In [13]:
print('\n--Average Actual vs Predicted Default Flag--')


--Average Actual vs Predicted Default Flag--


In [14]:
print('-Train Data-')
print(base_output_train_input_df[['actual_default_flag', 'predicted_default_flag']].agg('mean'))

-Train Data-
actual_default_flag       0.332331
predicted_default_flag    0.332420
dtype: float64


In [15]:
print('\n-Test Data-')
print(base_output_test_input_df[['actual_default_flag', 'predicted_default_flag']].agg('mean'))


-Test Data-
actual_default_flag       0.320000
predicted_default_flag    0.313658
dtype: float64


<h3>Force overfit</h3>

In [16]:
print('\n---Force Overfit Light GBM---')


---Force Overfit Light GBM---


<h4>Train model</h4>

In [17]:
model_parameters_dict = {
    'verbose': -1,
    'boosting_type': 'gbdt',
    'class_weight': None,
    'colsample_bytree': 1.0,
    'importance_type': 'split',
    'learning_rate': 0.1,
    'max_depth': -1,
    'min_child_samples': 1,
    'min_child_weight': 0.001,
    'min_split_gain': 0.0,
    'n_estimators': 100,
    'n_jobs': None,
    'num_leaves': 2000,
    'objective': 'binary',
    'random_state': 42,
    'reg_alpha': 0.0,
    'reg_lambda': 0.0,
    'subsample': 1.0,
    'subsample_for_bin': 200000,
    'subsample_freq': 0
}
force_overfit_light_gbm_model, _, _, _, _, force_overfit_output_train_input_df, force_overfit_output_test_input_df = LightGbmModelBuilder(train_input_df, test_input_df, independent_variables_list, target_variable_str, model_parameters_dict).run()

<h4>Get parameters</h4>

In [18]:
print('--Parameters--')
print(force_overfit_light_gbm_model.get_params())

--Parameters--
{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.1, 'max_depth': -1, 'min_child_samples': 1, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': None, 'num_leaves': 2000, 'objective': 'binary', 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbose': -1}


<h4>Compare average default flag between actual and predicted</h4>

In [19]:
print('\n--Average Actual vs Predicted Default Flag--')


--Average Actual vs Predicted Default Flag--


In [20]:
print('-Train Data-')
print(force_overfit_output_train_input_df[['actual_default_flag', 'predicted_default_flag']].agg('mean'))

-Train Data-
actual_default_flag       0.332331
predicted_default_flag    0.332331
dtype: float64


In [21]:
print('\n-Test Data-')
print(force_overfit_output_test_input_df[['actual_default_flag', 'predicted_default_flag']].agg('mean'))


-Test Data-
actual_default_flag       0.320000
predicted_default_flag    0.286057
dtype: float64


<h3>Parameter Optimisation</h3>

In [22]:
print('\n---Optimised Light GBM---')


---Optimised Light GBM---


<h4>Search for optimal parameters</h4>

In [23]:
model_parameters_dict = {
    'boosting_type': ['gbdt'],
    'colsample_bytree': [0.2, 0.5, 1.0],
    'importance_type': ['split'],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [-1, 10, 15],
    'min_child_samples': [1, 20, 30, 40],
    'min_child_weight': [0.001],
    'min_split_gain': [0.0],
    'n_estimators': [100],
    'num_leaves': [10, 31, 100, 2000],
    'objective': ['binary'],
    'random_state': [42],
    'reg_alpha': [0.0, 0.01, 0.05],
    'reg_lambda': [0.0, 0.01, 0.05],
    'subsample': [0.6, 0.8, 1.0],
    'subsample_for_bin': [100, 500, 2000, 200000],
    'subsample_freq': [0]
}

lgbm_model = lgbm.LGBMClassifier()
optimised_light_gbm_model = RandomizedSearchCV(lgbm_model, model_parameters_dict, n_iter=150, random_state=42, scoring='neg_log_loss', cv=2, verbose=0)
optimised_light_gbm_model.fit(X_train, y_train)

RandomizedSearchCV(cv=2, estimator=LGBMClassifier(), n_iter=150,
                   param_distributions={'boosting_type': ['gbdt'],
                                        'colsample_bytree': [0.2, 0.5, 1.0],
                                        'importance_type': ['split'],
                                        'learning_rate': [0.01, 0.05, 0.1],
                                        'max_depth': [-1, 10, 15],
                                        'min_child_samples': [1, 20, 30, 40],
                                        'min_child_weight': [0.001],
                                        'min_split_gain': [0.0],
                                        'n_estimators': [100],
                                        'num_leaves': [10, 31, 100, 2000],
                                        'objective': ['binary'],
                                        'random_state': [42],
                                        'reg_alpha': [0.0, 0.01, 0.05],
                                        'reg_lambda': [0.0, 0.01, 0.05],
                                        'subsample': [0.6, 0.8, 1.0],
                                        'subsample_for_bin': [100, 500, 2000,
                                                              200000],
                                        'subsample_freq': [0]},
                   random_state=42, scoring='neg_log_loss')

<h4>Get parameters</h4>

In [24]:
print('--Parameters--')
print(optimised_light_gbm_model.best_estimator_.get_params())

--Parameters--
{'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.05, 'max_depth': 15, 'min_child_samples': 40, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 100, 'n_jobs': None, 'num_leaves': 10, 'objective': 'binary', 'random_state': 42, 'reg_alpha': 0.01, 'reg_lambda': 0.01, 'subsample': 0.6, 'subsample_for_bin': 2000, 'subsample_freq': 0}


<h4>Compare average default flag between actual and predicted</h4>

In [25]:
print('\n--Average Actual vs Predicted Default Flag--')


--Average Actual vs Predicted Default Flag--


In [26]:
optimised_output_train_input_df = train_input_df.copy()
optimised_output_train_input_df['predicted_default_flag'] = optimised_light_gbm_model.best_estimator_.predict_proba(X_train)[:, 1]

print('-Train Data-')
print(optimised_output_train_input_df[['actual_default_flag', 'predicted_default_flag']].agg('mean'))

-Train Data-
actual_default_flag       0.332331
predicted_default_flag    0.332389
dtype: float64


In [27]:
optimised_output_test_input_df = test_input_df.copy()
optimised_output_test_input_df['predicted_default_flag'] = optimised_light_gbm_model.best_estimator_.predict_proba(X_test)[:, 1]

print('\n-Test Data-')
print(optimised_output_test_input_df[['actual_default_flag', 'predicted_default_flag']].agg('mean'))


-Test Data-
actual_default_flag       0.320000
predicted_default_flag    0.313327
dtype: float64


In [28]:
print('\n--End Light GBM Model Build--\n')


--End Light GBM Model Build--



<h2>Output</h2>
<p>Output model</p>

In [29]:
output_folder_path = rf"{cwd}/{theme}/data/outputs/01_lgbm_introduction"

In [30]:
output_file_path = rf"{output_folder_path}/base_light_gbm_model.gz"
print(rf"Output File (Base Light GBM Model): {output_file_path}")
joblib.dump(base_light_gbm_model, output_file_path)

Output File (Base Light GBM Model): c:\dev\amts/machine_learning/data/outputs/01_lgbm_introduction/base_light_gbm_model.gz


['c:\\dev\\amts/machine_learning/data/outputs/01_lgbm_introduction/base_light_gbm_model.gz']

In [31]:
output_file_path = rf"{output_folder_path}/force_overfit_light_gbm_model.gz"
print(rf"Output File (Force Overfit Light GBM Model): {output_file_path}")
joblib.dump(force_overfit_light_gbm_model, output_file_path)

Output File (Force Overfit Light GBM Model): c:\dev\amts/machine_learning/data/outputs/01_lgbm_introduction/force_overfit_light_gbm_model.gz


['c:\\dev\\amts/machine_learning/data/outputs/01_lgbm_introduction/force_overfit_light_gbm_model.gz']

In [32]:
output_file_path = rf"{output_folder_path}/optimised_light_gbm_model.gz"
print(rf"Output File (Optimised Light GBM Model): {output_file_path}")
joblib.dump(optimised_light_gbm_model.best_estimator_, output_file_path)

Output File (Optimised Light GBM Model): c:\dev\amts/machine_learning/data/outputs/01_lgbm_introduction/optimised_light_gbm_model.gz


['c:\\dev\\amts/machine_learning/data/outputs/01_lgbm_introduction/optimised_light_gbm_model.gz']